# The oscillating double — the obvious suspect

Notebook 01 showed a naive DQN plateaus ~13 points below the table and no single knob moves it. The
most visible instability in those runs is an **oscillating value on the terminal `double` action**
that never settles — so the final policy is a snapshot of a value still in motion. This notebook
characterizes that oscillation and separates two candidate mechanisms.

It is the obvious suspect for the plateau, and we treat it that way. Notebook 03 chases the fix and
finds — importantly — that curing the oscillation does *not* close the gap. So read this as a symptom
to understand precisely, not yet the cause of the cap.

In [ ]:
import sys; sys.path.insert(0, '.')
import statistics as st
import matplotlib.pyplot as plt
from runs import load_runs, probe_trajectory, sample_counts

df = load_runs(); dqn = df[df.method == 'dqn']
osc = dqn[(dqn.encoding=='onehot') & (dqn.lr_schedule=='constant') & dqn.has_qgrid
          & (dqn.episodes==1_000_000) & (dqn.reward_baseline=='none') & (dqn.double_after==0)
          & (dqn.target_tau==0.0) & (~dqn.swa)].iloc[0]
soft = dqn[dqn.target_tau > 0].iloc[0]
print('oscillation reference:', osc.run, '| soft-target run:', soft.run)

## The symptom: `Q(double)` never settles

Watched over training, the probe cell *soft-20 vs dealer-8*: `Q(stand)` is stable, but `Q(double)`
swings ±1–2 the whole run. Doubling is **terminal** (one card, hand resolves) and high-variance (a ±2
stake on a near-coin-flip), and the estimate never converges — the final agreement is just where that
swing happened to be at the last checkpoint.

In [ ]:
ep, q_dbl = probe_trajectory(osc.path, 'soft20_v8', 'double')
_,  q_std = probe_trajectory(osc.path, 'soft20_v8', 'stand')
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ep, q_dbl, color='#D85A30', lw=1.5, label='Q(double)')
ax.plot(ep, q_std, color='#1D9E75', lw=1.5, label='Q(stand)')
ax.set_xlabel('episode'); ax.set_ylabel('Q'); ax.set_title('soft-20 v8: double oscillates, stand is stable')
ax.legend(); plt.tight_layout(); plt.show()
print(f'back-half std  Q(double)={st.pstdev(q_dbl[len(q_dbl)//2:]):.2f}  vs  Q(stand)={st.pstdev(q_std[len(q_std)//2:]):.2f}')

## It isn't raw under-sampling

The natural first guess is that the cell is starved of data. It isn't. The oscillating soft-20 v8 is
visited well over a thousand times; and where the agent actively *over-doubles* — soft-20 v6, whose
correct play is **stand** — it samples `double` *more* than `stand` and still lands on the wrong
action. Hundreds-to-thousands of samples, still oscillating or still wrong: the problem is how the
value is averaged, not how much data it sees. (The clincher is in 03 — a decaying step settles the
same cell on the same data budget.)

In [ ]:
sc = sample_counts(osc.path)
print(f"{'cell':36}{'double':>8}{'stand':>8}{'hit':>8}")
for label, (v, soft_, u) in [('soft-20 v8  (stand cell, oscillates)', (20, True, 8)),
                             ('soft-20 v6  (over-doubled vs truth)',  (20, True, 6)),
                             ('hard-11 v6  (correct double)',         (11, False, 6))]:
    print(f'{label:36}{sc.get((v,soft_,u,"double"),0):>8,}{sc.get((v,soft_,u,"stand"),0):>8,}{sc.get((v,soft_,u,"hit"),0):>8,}')

## Two mechanisms, told apart by experiment

- **Mechanism 1 — feedback loop:** the estimate picks the action, which picks the data, which feeds
  back. Damped by anything that stabilizes the bootstrap or the data.
- **Mechanism 2 — constant-gain variance:** a running mean of a high-variance reward under a
  *constant* learning rate has non-vanishing steady-state variance — it converges to a distribution,
  not a point.

A **soft/EMA target** damps Mechanism 1. If the double were a feedback-loop problem it would settle.
It doesn't — because `double` is *terminal* (target = raw reward; the target network isn't in its
path). That isolates Mechanism 2. (Exploring starts, which forces uniform data, likewise fails to
settle it — same conclusion; see notebook 04.)

In [ ]:
_, q_const = probe_trajectory(osc.path,  'soft20_v8', 'double')
_, q_soft  = probe_trajectory(soft.path, 'soft20_v8', 'double')
print('soft-20 v8  Q(double) back-half std:')
print(f'  constant lr     : {st.pstdev(q_const[len(q_const)//2:]):.2f}')
print(f'  soft/EMA target : {st.pstdev(q_soft[len(q_soft)//2:]):.2f}   (still oscillating -> not the loop)')

## Conclusion

The naive runs carry an oscillating value on the terminal `double`: a high-variance reward learned
with a constant-gain step, so the estimate never converges. It is *not* under-sampling and *not* the
feedback loop. The tabular agent avoided it because its `1/n` step converges; the network's constant
Adam lr does not. The obvious fix is to make the step decay — which notebook 03 does.

But hold the conclusion loosely. In 03 the decay settles the oscillation **completely** and agreement
barely moves — which exposes this oscillation as a *symptom*, not the thing capping agreement. A
variance we could see and fix turns out not to be the binding constraint; the real cause waits in
notebooks 04–05.